In [ ]:
import sys
import os

module_directory = os.path.abspath('code/')

# Add the directory to the Python path
sys.path.append(module_directory)

from inference import model_fn, transform_fn
m = model_fn("model")
body = json.dumps({"inputs": "A quick test sentence."})
out, ctype = transform_fn(m, body, "application/json", "application/json")
print(ctype, out[:80], "...")


In [ ]:

import json, math, sys, re
from typing import Any, Iterable, Tuple, Optional

# --- core checks ---
CONTROL_CHARS_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")  # raw control chars (must be escaped in JSON text)

def _check_numbers(x: Any, path: str="$") -> Optional[Tuple[str, float]]:
    """Return (path, value) if a non-finite number is found anywhere."""
    if isinstance(x, float):
        if not math.isfinite(x):
            return (path, x)
    elif isinstance(x, (list, tuple)):
        for i, v in enumerate(x):
            hit = _check_numbers(v, f"{path}[{i}]")
            if hit: return hit
    elif isinstance(x, dict):
        for k, v in x.items():
            hit = _check_numbers(v, f"{path}.{k}")
            if hit: return hit
    return None

def _check_schema(obj: Any) -> str:
    """Return an error string or '' if OK for schema:
       {"doc_id": str, "sent_id": int, "inputs": str}"""
    if not isinstance(obj, dict):
        return "record is not a JSON object"
    for k in ("docid", "sentid", "inputs"):
        if k not in obj:
            return f"missing required key: {k}"
    if not isinstance(obj["docid"], str) or not obj["docid"]:
        return "docid must be a non-empty string"
    if not isinstance(obj["sentid"], int):
        return "sentid must be an integer"
    if not isinstance(obj["inputs"], str) or not obj["inputs"]:
        return "inputs must be a non-empty string"
    if CONTROL_CHARS_RE.search(obj["docid"]):
        return "docid contains raw control characters"
    # (inputs may legitimately contain \n, \t etc as escaped sequences in JSON; the decoder handles that)
    return ""

def validate_jsonl_lines(lines: Iterable[bytes]) -> None:
    """Validate a JSONL stream (bytes per line). Prints first problem found and exits(1); otherwise prints OK."""
    # Detect BOM on very first bytes
    first = True
    for i, raw in enumerate(lines, 1):
        if not raw:
            # skip blank lines; JSON Lines allows them but many pipelines don’t expect them
            continue
        if first:
            first = False
            if raw.startswith(b"\xef\xbb\xbf"):  # UTF-8 BOM
                print(f"Line {i}: UTF-8 BOM detected; prefer UTF-8 without BOM.", file=sys.stderr)

        # Strict UTF-8; fail fast on decoding errors
        try:
            s = raw.decode("utf-8", "strict")
        except UnicodeDecodeError as e:
            print(f"Line {i}: UTF-8 decode error: {e}", file=sys.stderr)
            sys.exit(1)

        # Must be a single JSON value per line (no trailing commas etc.)
        try:
            rec = json.loads(s)
        except json.JSONDecodeError as e:
            head = s[:120].replace("\n", "\\n")
            print(f"Line {i}: invalid JSON ({e.msg}) at pos {e.pos}. Head: {head!r}", file=sys.stderr)
            sys.exit(1)

        # Schema & numeric sanity
        err = _check_schema(rec)
        if err:
            print(f"Line {i}: {err}", file=sys.stderr)
            sys.exit(1)

        hit = _check_numbers(rec)
        if hit:
            path, val = hit
            print(f"Line {i}: non-finite number at {path}: {val!r} (JSON forbids NaN/Infinity).", file=sys.stderr)
            sys.exit(1)

    print("Input looks good: UTF-8 OK, valid JSON Lines, schema OK, no NaN/Infinity found.")

# --- usage examples ---

# 1) Local file:
# with open("your_input.jsonl", "rb") as f:
#     validate_jsonl_lines(f)

# 2) S3 object:
# import boto3
# s3 = boto3.client("s3")
# obj = s3.get_object(Bucket="your-bucket", Key="path/to/input.jsonl")
# validate_jsonl_lines(obj["Body"].iter_lines())


import boto3, json

key = f"{prefix}/batch-output/{run_id}.jsonl"

s3 = boto3.client("s3")
obj = s3.get_object(Bucket=bucket, Key=key)
validate_jsonl_lines(obj["Body"].iter_lines())



In [1]:
%run testing_metrics.py

file paths assigned

=== paths assigned ===
Documents loaded
The positive text:
 With regard to the Commission’s claim that the first ground of appeal is inadmissible, it should be noted that that ground, which forms part of the submissions relating to the existence of State aid, was put forward at first instance in the context of the first and second pleas relied on before the General Court.
The positive text:
 In those circumstances, the General Court was correct to hold, at paragraph 323 of the judgment under appeal, that the finding of the existence of aid depended on a number of ‘circumstances unrelated to’ the special tax regime, such as the fact that the business tax was charged annually and the level of the tax rates voted each year by the authorities in the territory in which FT had establishments.
It passed positive assignation

=== Macro metrics (same-document ranking) ===
Hit@ 1 = 0.0807
Hit@ 5 = 0.2153
Hit@10 = 0.3147
Hit@20 = 0.4555
Hit@50 = 0.6729
MRR     = 0.1596
Mean r

In [5]:
0.0196*(102+88+144+148)/50

0.188944

In [2]:
[21.49/115, 25.3/147, 63.35/248,60.92/206]

[0.1868695652173913,
 0.17210884353741496,
 0.25544354838709676,
 0.29572815533980584]

In [3]:
1333.9/3888

0.3430812757201646

In [1]:
1/241

0.004149377593360996

In [2]:
5/241.5

0.020703933747412008

In [3]:
10/241.5

0.041407867494824016

In [4]:
20/241.5

0.08281573498964803

In [5]:
50/241.5

0.2070393374741201